In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader

# 1. 공백이나 특수문자 에러를 방지하기 위해 절대 경로 또는 정확한 상대 경로 세팅
file_path = "data/공무원보수규정(대통령령)(제36501호)(20260707).pdf"

# 2. 파이썬이 이 파일을 인식하고 있는지 먼저 확인
if not os.path.exists(file_path):
    print(f"❌ [에러] 파일을 찾을 수 없습니다! 현재 경로: {os.getcwd()}")

else:
    print("p 파일이 존재합니다! 로드를 시작합니다.")
    
    # 3. 파일이 존재할 때만 로더 실행
    loader = PyPDFLoader(file_path)
    docs = loader.load_and_split()

C:\Users\박동관\AppData\Local\Temp\ipykernel_54232\17869605.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


p 파일이 존재합니다! 로드를 시작합니다.


In [8]:
import re
from langchain_core.documents import Document
import os
from langchain_community.document_loaders import PyPDFLoader

# 파일별 법령명 매핑 (문서 상단 표제를 보고 직접 지정 — 가장 확실한 방법)
LAW_FILES = {
    "data/공무원보수규정(대통령령)(제36501호)(20260707).pdf": "공무원보수규정",
}

# "제3조의2(공무원처우 개선계획)" 도 잡도록 (의\d+) 추가
article_pattern = r"(제\s*\d+\s*조(?:\s*의\s*\d+)?(?:\([^\)]+\))?)"

def parse_law_to_docs(file_path: str, law_name: str, docs) -> list[Document]:
    full_text = "\n".join([d.page_content for d in docs])
    parts = re.split(article_pattern, full_text)

    chunked = []
    current_title = None
    current_number = None

    for item in parts:
        if not item or not item.strip():
            continue
        item = item.strip()

        if re.fullmatch(article_pattern, item):
            current_title = item
            # "제14조(승급의 제한)" -> number="제14조", title="승급의 제한"
            num_match = re.match(r"(제\s*\d+\s*조(?:\s*의\s*\d+)?)", item)
            current_number = re.sub(r"\s+", "", num_match.group(1)) if num_match else item
            continue

        if current_title is None:
            continue  # 조문 시작 전 머릿말(제목/시행일 등) 스킵

        content = item.strip()
        if len(content) > 10:
            chunked.append(Document(
                page_content=f"{current_title} {content}",
                metadata={
                    "law_name": law_name,
                    "article_number": current_number,   # "제14조" — ID로 씀
                    "section": current_title,
                }
            ))
    return chunked

# 실행: 파일마다 로드 → 파싱 → 합치기
all_chunked_docs = []
for file_path, law_name in LAW_FILES.items():
    loader = PyPDFLoader(file_path)
    docs = loader.load_and_split()  # 기존에 쓰던 PDF 로더 그대로
    law_chunks = parse_law_to_docs(file_path, law_name, docs)
    print(f"📄 {law_name}: {len(law_chunks)}개 조문 파싱됨")
    all_chunked_docs.extend(law_chunks)

print(f"\n✂️ 전체 {len(all_chunked_docs)}개 조문 청크 완성")

📄 공무원보수규정: 250개 조문 파싱됨

✂️ 전체 250개 조문 청크 완성


In [11]:
all_chunked_docs[110]

Document(metadata={'law_name': '공무원보수규정', 'article_number': '제36조의2', 'section': '제36조의2'}, page_content='제36조의2 제1항에 따른 국립대학의 교원은 제외한다)에게는 8만\n2800원을 근속가봉으로 지급하되, 가산하는 횟수는 10회를 초과하지 못한다.<개정 2011. 1. 10., 2012. 1. 6., 2013.\n1. 9., 2014. 1. 8., 2015. 1. 6., 2016. 1. 8., 2017. 1. 6., 2018. 1. 18., 2019. 1. 8., 2020. 1. 7., 2021. 1. 5., 2022. 1. 4.,\n2023. 1. 6., 2024. 1. 5., 2025. 1. 3., 2026. 1. 2.>\n③ 군인(별표 13의 봉급표를 적용받는 공무원을 말한다)에게는 계급별 승급액에 해당하는 금액을 근속가봉으로 지\n급한다. 다만, 대위ㆍ중위ㆍ중사 및 하사의 경우 최종 승급액을 기준으로 한다.<개정 2010. 1. 7., 2018. 1. 18., 2019.\n1. 8., 2024. 1. 5., 2025. 1. 3.>\n④ 삭제<2010. 1. 7.>\n[전문개정 2009. 3. 31.]')

In [20]:
import os
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import time

load_dotenv()
graph = Neo4jGraph(
    url=os.getenv('NEO4J_URL'),
    username=os.getenv('NEO4J_USERNAME'),
    password=os.getenv('NEO4J_PASSWORD'),
)

print("🧹 DB 초기화 및 제약조건 설정...")
graph.query("MATCH (n) DETACH DELETE n")
# ID 중복 방지를 위한 유니크 제약조건 (이것만 남김)
graph.query("CREATE CONSTRAINT article_id IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE")

# ── 1단계: LLM 설정 및 고정 스키마 정의 ──
llm = ChatOpenAI(model="gpt-4o", temperature=0)

allowed_nodes = ["Law", "Article", "Term", "PersonnelCategory", "Regulation", 
                 "DisciplineType", "Committee", "Organization", "Rate", "Condition"]

allowed_relationships = [
    ("Law", "HAS_ARTICLE", "Article"),
    ("Article", "DEFINES", "Term"),
    ("Article", "APPLIES_TO", "PersonnelCategory"),
    ("Article", "REGULATES", "Regulation"),
    ("Article", "REGULATES", "DisciplineType"),
    ("Article", "ESTABLISHES", "Committee"),
    ("Article", "SPECIFIES_RATE", "Rate"),
    ("Rate", "UNDER_CONDITION", "Condition"),
    ("Article", "CITES", "Article"),       
    ("Article", "DELEGATES_TO", "Article"), 
]

# 지침(Prompt)을 명확히 주면 LLM이 알아서 관계를 잘 엮습니다.
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=["name", "description"],
    strict_mode=True,
    # 중괄호를 {{ }} 로 감싸서 변수가 아닌 '문자'임을 LangChain에게 알려줍니다.
    additional_instructions="""
당신은 법률 문서를 지식 그래프로 변환하는 전문가입니다. 아래 규칙을 반드시 따르세요.

1. [가장 중요] 중심 조문(Article) 노드의 ID 생성 규칙:
   - 문서 메타데이터의 'law_name'과 'article_number'를 조합하여 반드시 'law_name::article_number' 형태로 ID를 지정하십시오. (예: '공무원보수규정::제1조')
   - 해당 조문이 속한 Law 노드를 생성하고, (Law)-[:HAS_ARTICLE]->(Article) 관계를 반드시 연결하십시오.

2. CITES와 DELEGATES_TO 규칙:
   - "「군인보수법」 제8조제2항에 따라"처럼 다른 법령/조문을 명시 인용한 경우: CITES
   - "대통령령으로 정한다"처럼 하위법령에 위임하는 표현: DELEGATES_TO
   - 인용/위임 대상 조문의 ID도 알 수 있다면 '해당법령명::조번호' 형태로 작성하되, 법령명을 모르면 조번호만 적으십시오. 절대 추측하지 마세요.

3. 본문에 확실한 근거가 없는 개체와 관계는 절대 생성하지 마십시오.
""",
)

print("🤖 의미관계 추출 및 Neo4j 적재 시작...")

# TPM 제한(30,000)이 낮으므로, 한 번에 2~3개씩 안전하게 처리합니다.
CHUNK_SIZE = 5 
for i in range(0, len(all_chunked_docs), CHUNK_SIZE):
    chunk = all_chunked_docs[i:i + CHUNK_SIZE]
    print(f"📦 처리 중: {i} ~ {min(i + CHUNK_SIZE, len(all_chunked_docs))} / {len(all_chunked_docs)}")
    
    # ⚠️ API 에러 발생 시 멈추지 않고 재시도하도록 예외 처리 추가
    try:
        graph_documents = transformer.convert_to_graph_documents(chunk)
        
        if graph_documents:
            graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=False)
            
        # API 과부하 방지를 위해 한 턴 끝나고 2초간 휴식
        time.sleep(2)
        
    except Exception as e:
        if "429" in str(e) or "rate_limit" in str(e).lower():
            print("⏳ Rate Limit 감지! 15초간 대기 후 재시도합니다...")
            time.sleep(5)
            # 한 번 더 시도
            graph_documents = transformer.convert_to_graph_documents(chunk)
            if graph_documents:
                graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=False)
        else:
            print(f"❌ 예상치 못한 에러 발생: {e}")
            raise e

print("\n🎉 안전하게 적재 완료!")

🧹 DB 초기화 및 제약조건 설정...
🤖 의미관계 추출 및 Neo4j 적재 시작...
📦 처리 중: 0 ~ 5 / 250


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=108, offset=107>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 1, 'column': 108}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties WITH source, row CALL apoc.create.addLabels( source, [row.type] ) YIELD node RETURN distinct 'done' AS result"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn:

📦 처리 중: 5 ~ 10 / 250


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=108, offset=107>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 1, 'column': 108}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties WITH source, row CALL apoc.create.addLabels( source, [row.type] ) YIELD node RETURN distinct 'done' AS result"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn:

📦 처리 중: 10 ~ 15 / 250


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=108, offset=107>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 1, 'column': 108}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties WITH source, row CALL apoc.create.addLabels( source, [row.type] ) YIELD node RETURN distinct 'done' AS result"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn:

📦 처리 중: 130 ~ 135 / 250


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=108, offset=107>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 1, 'column': 108}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties WITH source, row CALL apoc.create.addLabels( source, [row.type] ) YIELD node RETURN distinct 'done' AS result"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn:

📦 처리 중: 135 ~ 140 / 250
⏳ Rate Limit 감지! 15초간 대기 후 재시도합니다...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-w8e04fAdHmB76aYlooKL00Dt on tokens per min (TPM): Limit 30000, Used 29904, Requested 863. Please try again in 1.534s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}